# EfficientNet V2 + SAM Masking

Pipeline: SAM wycina świnie z tła → EfficientNet V2 klasyfikuje pozy.

Ulepszenia vs baseline:
- **SAM** — usuwanie tła, model widzi tylko świnię
- **Train1 + Train2** — ~2x więcej danych treningowych
- **Label smoothing** — mniej overfittingu
- **TTA** — Test-Time Augmentation przy inferencji
- **Checkpointy** — zapis modelu po każdej epoce

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

In [ ]:
!kaggle competitions download -c multi-view-pig-posture-recognition

In [ ]:
!unzip -q multi-view-pig-posture-recognition.zip -d data

In [ ]:
!ls data/multiview_pig_posture_recognition

## Instalacja SAM (Segment Anything Model)

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python-headless

# Pobierz wagi SAM ViT-B (najszybszy wariant, dobry balans speed/quality)
import os
if not os.path.exists("sam_vit_b_01ec64.pth"):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print("Pobrano wagi SAM ViT-B")
else:
    print("Wagi SAM już istnieją")

## Importy

In [ ]:
import os, ast, copy, numpy as np, pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from segment_anything import sam_model_registry, SamPredictor

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

## Konfiguracja

In [ ]:
CLASS_NAMES = {
    0: "Lateral_lying_left",
    1: "Lateral_lying_right",
    2: "Sitting",
    3: "Standing",
    4: "Sternal_lying",
}

BASE_DIR   = "data/multiview_pig_posture_recognition"
BATCH_SIZE = 32
EPOCHS     = 6
IMG_SIZE   = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Wczytanie danych — Train1 + Train2 + Test

In [ ]:
train1 = pd.read_csv(os.path.join(BASE_DIR, "train1.csv"))
train2 = pd.read_csv(os.path.join(BASE_DIR, "train2.csv"))
test   = pd.read_csv(os.path.join(BASE_DIR, "test.csv"))

train1["source"] = "train1"
train2["source"] = "train2"
test["source"]   = "test"

for df in [train1, train2, test]:
    df["bbox_parsed"] = df["bbox"].apply(ast.literal_eval)

train1["class_name"] = train1["class_id"].map(CLASS_NAMES)
train2["class_name"] = train2["class_id"].map(CLASS_NAMES)

# Łączymy Train1 + Train2
train_df = pd.concat([train1, train2], ignore_index=True)
print("Train1:", len(train1), "| Train2:", len(train2), "| Łącznie:", len(train_df))
print("Test:", len(test))
print()
print(train_df["class_name"].value_counts())

## Funkcje pomocnicze

In [ ]:
def load_image(image_id, source):
    folder_map = {
        "train1": "train1_images",
        "train2": "train2_images",
        "test":   "test_images",
    }
    path = os.path.join(BASE_DIR, folder_map[source], image_id)
    return Image.open(path).convert("RGB")


def get_bbox_xyxy(bbox_parsed, img_w, img_h, padding=0.12):
    """Zwraca bbox w formacie xyxy z paddingiem, przyciętym do granic obrazu."""
    x, y, w, h = map(float, bbox_parsed)
    pad_x, pad_y = w * padding, h * padding
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(img_w, x + w + pad_x)
    y2 = min(img_h, y + h + pad_y)
    return np.array([x1, y1, x2, y2])


def crop_square(image_pil, bbox_parsed, padding=0.12):
    """Standardowy crop kwadratowy (fallback gdy SAM nie dostępny)."""
    img_w, img_h = image_pil.size
    x, y, w, h = map(float, bbox_parsed)
    pad_x, pad_y = w * padding, h * padding
    x1, y1 = x - pad_x, y - pad_y
    x2, y2 = x + w + pad_x, y + h + pad_y
    side = max(x2 - x1, y2 - y1)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    x1, y1 = cx - side/2, cy - side/2
    x2, y2 = cx + side/2, cy + side/2
    x1 = max(0, int(round(x1))); y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2))); y2 = min(img_h, int(round(y2)))
    return image_pil.crop((x1, y1, x2, y2))

## Inicjalizacja SAM

In [ ]:
# Ładujemy SAM ViT-B
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam = sam.to(device)
sam_predictor = SamPredictor(sam)

print("SAM załadowany na:", device)

## Funkcja cropowania z maską SAM

SAM dostaje bounding box → generuje maskę świni → tło zamieniane na szary kolor → crop kwadratowy.

In [ ]:
def crop_with_sam_mask(image_pil, bbox_parsed, sam_predictor, padding=0.12, bg_color=128):
    """
    Wycina świnię z tłem usuniętym przez SAM.
    Tło zastępowane kolorem bg_color (domyślnie szary ~ImageNet mean).
    Fallback na standardowy crop jeśli SAM zawiedzie.
    """
    img_np = np.array(image_pil)
    img_w, img_h = image_pil.size

    try:
        bbox_xyxy = get_bbox_xyxy(bbox_parsed, img_w, img_h, padding=0.0)

        sam_predictor.set_image(img_np)
        masks, scores, _ = sam_predictor.predict(
            box=bbox_xyxy,
            multimask_output=True,
        )
        # Wybierz maskę z najwyższym score
        best_mask = masks[np.argmax(scores)]  # (H, W) bool

        # Zastąp tło szarym kolorem
        masked = img_np.copy()
        masked[~best_mask] = bg_color

        # Crop kwadratowy z paddingiem
        x, y, w, h = map(float, bbox_parsed)
        pad_x, pad_y = w * padding, h * padding
        x1, y1 = x - pad_x, y - pad_y
        x2, y2 = x + w + pad_x, y + h + pad_y
        side = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, y1 = int(max(0, cx - side/2)), int(max(0, cy - side/2))
        x2, y2 = int(min(img_w, cx + side/2)), int(min(img_h, cy + side/2))

        crop = masked[y1:y2, x1:x2]
        return Image.fromarray(crop)

    except Exception:
        # Fallback: standardowy crop bez maski
        return crop_square(image_pil, bbox_parsed, padding=padding)


def visualize_sam_crops(df, n=4):
    """Pokaż porównanie: oryginalny crop vs crop z maską SAM."""
    samples = df.sample(n=n, random_state=42)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))

    for i, (_, row) in enumerate(samples.iterrows()):
        img = load_image(row["image_id"], row["source"])

        # Oryginalny crop
        orig = crop_square(img, row["bbox_parsed"])
        axes[0, i].imshow(orig)
        axes[0, i].set_title(f"Oryginalny\n{CLASS_NAMES[row['class_id']]}", fontsize=9)
        axes[0, i].axis("off")

        # SAM crop
        sam_crop = crop_with_sam_mask(img, row["bbox_parsed"], sam_predictor)
        axes[1, i].imshow(sam_crop)
        axes[1, i].set_title("SAM masked", fontsize=9)
        axes[1, i].axis("off")

    plt.suptitle("Porównanie: oryginalny crop vs SAM masked crop", fontsize=14)
    plt.tight_layout()
    plt.show()

print("Funkcje SAM gotowe. Uruchom visualize_sam_crops(train_df) żeby zobaczyć efekt.")

In [ ]:
# Podgląd efektu SAM na kilku próbkach
visualize_sam_crops(train_df, n=4)

## Dataset z SAM maskowaniem

In [ ]:
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.05, p=0.3):
        self.mean, self.std, self.p = mean, std, p
    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = torch.clamp(tensor + torch.randn_like(tensor)*self.std + self.mean, 0, 1)
        return tensor


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=90),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    AddGaussianNoise(p=0.2, std=0.03),
    transforms.RandomErasing(p=0.2, scale=(0.01, 0.05), ratio=(0.75, 1.33), value="random"),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class PigPostureDataset(Dataset):
    """
    Dataset z opcjonalnym SAM maskowaniem.
    use_sam=True  → usuwa tło przy każdym cropie (wolniejsze, lepsze)
    use_sam=False → standardowy crop kwadratowy (szybkie)
    """
    def __init__(self, df, transform=None, is_test=False, use_sam=True, sam_predictor=None):
        self.df            = df.reset_index(drop=True)
        self.transform     = transform
        self.is_test       = is_test
        self.use_sam       = use_sam and (sam_predictor is not None)
        self.sam_predictor = sam_predictor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = load_image(row["image_id"], row["source"])

        if self.use_sam:
            crop = crop_with_sam_mask(image, row["bbox_parsed"], self.sam_predictor)
        else:
            crop = crop_square(image, row["bbox_parsed"])

        if self.transform:
            crop = self.transform(crop)

        if self.is_test:
            return crop, row["row_id"]

        return crop, int(row["class_id"])

## DataLoaders

> **Uwaga:** SAM działa na CPU/GPU sekwencyjnie — ustaw `num_workers=0` żeby uniknąć konfliktów z CUDA w wielu procesach.

In [ ]:
# WeightedRandomSampler — balansowanie klas
train_targets  = train_df["class_id"].values
class_counts   = np.bincount(train_targets, minlength=len(CLASS_NAMES))
class_weights  = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([class_weights[y] for y in train_targets])

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

print("Class counts:", dict(zip(CLASS_NAMES.values(), class_counts)))

In [ ]:
# num_workers=0 bo SAM używa GPU i nie działa w fork procesach
train_dataset = PigPostureDataset(
    train_df, transform=train_transform,
    use_sam=True, sam_predictor=sam_predictor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,   # ważne przy SAM
    pin_memory=False,
)

print(f"Train batches: {len(train_loader)}")

# Sanity check
imgs, lbls = next(iter(train_loader))
print("Batch shape:", imgs.shape, "| Labels:", lbls[:8].tolist())

## Model — EfficientNet V2 S z label smoothing

In [ ]:
num_classes = len(CLASS_NAMES)

weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
model   = models.efficientnet_v2_s(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

# Label smoothing — zapobiega overfittingowi, lepsza generalizacja
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parametry modelu: {total_params:,}")
print(f"Label smoothing: 0.1")

## Trening z checkpointami

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (total_loss / len(loader.dataset),
            accuracy_score(all_labels, all_preds),
            f1_score(all_labels, all_preds, average="macro"))

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
history = []

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader)
    scheduler.step()

    history.append({"epoch": epoch+1, "loss": train_loss, "acc": train_acc, "f1": train_f1})

    print(f"Epoch {epoch+1}/{EPOCHS} | loss {train_loss:.4f} | acc {train_acc:.4f} | f1 {train_f1:.4f}")

    # Checkpoint po każdej epoce
    torch.save({
        "epoch":             epoch + 1,
        "model_state_dict":  model.state_dict(),
        "optimizer_state":   optimizer.state_dict(),
        "train_f1":          train_f1,
    }, f"checkpoints/epoch_{epoch+1}.pt")

print("\nTrening zakończony. Checkpointy zapisane w /checkpoints/")

In [ ]:
# Wykres treningu
epochs_list = [h["epoch"] for h in history]
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_list, [h["loss"] for h in history], marker="o")
plt.title("Train Loss"); plt.xlabel("Epoch"); plt.grid(True)
plt.subplot(1, 2, 2)
plt.plot(epochs_list, [h["f1"] for h in history], marker="o", color="green")
plt.title("Train F1 (macro)"); plt.xlabel("Epoch"); plt.grid(True)
plt.tight_layout(); plt.show()

### Wybierz epokę do inferencji

Sprawdź wyniki na Kaggle po każdej epoce. Zwykle epoka 4-5 generalizuje lepiej niż ostatnia.

In [ ]:
# ── Zmień tę wartość na epokę którą chcesz użyć do submisji ──
BEST_EPOCH = 5

checkpoint = torch.load(f"checkpoints/epoch_{BEST_EPOCH}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Załadowano model z epoki {BEST_EPOCH} (train_f1={checkpoint['train_f1']:.4f})")

In [ ]:
# Pobierz checkpointy na laptop (opcjonalne — możesz pobrać konkretną epokę)
from google.colab import files

# Pobierz wybraną epokę
files.download(f"checkpoints/epoch_{BEST_EPOCH}.pt")

## Inference z Test-Time Augmentation (TTA)

Dla każdego obrazu robimy predykcję 3 razy (oryginał + flip poziomy + flip pionowy) i uśredniamy softmax.

In [ ]:
# Test dataset z SAM
test_dataset = PigPostureDataset(
    test, transform=val_transform,
    is_test=True, use_sam=True, sam_predictor=sam_predictor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print("Test batches:", len(test_loader))

In [ ]:
def predict_with_tta(model, loader, device):
    """
    TTA: uśrednia predykcje z 3 wariantów:
    - oryginalny obraz
    - flip poziomy
    - flip pionowy
    """
    model.eval()
    all_row_ids = []
    all_probs   = []

    with torch.no_grad():
        for images, ids in tqdm(loader, desc="TTA Inference"):
            images = images.to(device)

            # 3 warianty TTA
            flip_h = torch.flip(images, dims=[3])
            flip_v = torch.flip(images, dims=[2])

            p1 = F.softmax(model(images),  dim=1)
            p2 = F.softmax(model(flip_h),  dim=1)
            p3 = F.softmax(model(flip_v),  dim=1)

            avg_probs = (p1 + p2 + p3) / 3.0

            all_probs.extend(avg_probs.cpu().numpy())
            all_row_ids.extend(ids)

    return all_row_ids, all_probs


row_ids, probs = predict_with_tta(model, test_loader, device)
predictions = [int(np.argmax(p)) for p in probs]

print("Predykcje gotowe:", len(predictions))

## Zapis submission

In [ ]:
submission = pd.DataFrame({
    "row_id":   row_ids,
    "class_id": predictions,
})

assert len(submission) == len(test), f"Błąd: {len(submission)} != {len(test)}"
assert set(submission["class_id"].unique()).issubset({0,1,2,3,4}), "Błędne klasy!"

fname = f"T2_efficientnetv2_sam_tta_ep{BEST_EPOCH}.csv"
submission.to_csv(fname, index=False)

print(f"Zapisano: {fname}")
print(submission["class_id"].value_counts().sort_index())
submission.head()

In [ ]:
from google.colab import files
files.download(fname)